# Create Materialized Lakehouse View: `gold_scored_policy_period`

Combines the four **policy-period grain** gold tables into a single wide, analytics-ready Delta table:

| Source Table | Contribution |
|---|---|
| `gold_policy_period_features` | UBI driving features, vehicle info, current premium |
| `gold_policy_period_loss` | Actual claims & payouts (calibration truth) |
| `gold_expected_loss_scores` | ML risk score & expected loss cost |
| `gold_policy_premium_recommendation` | Recommended premium, pricing loads, guardrails |

**Join key:** `(policy_number, policy_start_date, policy_end_date)`

`gold_policy_period_features` is the anchor (INNER or LEFT joins from there depending on data availability).

Duplicate columns across tables (e.g. `coverage_type`, `current_premium`, `expected_loss_cost`, `feature_version`) are resolved by taking the most authoritative source and aliasing where needed.

In [1]:
# -------------------------------------------------------------------
# Cell 2 – Read the four source gold tables
# -------------------------------------------------------------------
df_features = spark.table("gold_policy_period_features")
df_loss     = spark.table("gold_policy_period_loss")
df_scores   = spark.table("gold_expected_loss_scores")
df_premium  = spark.table("gold_policy_premium_recommendation")

print("Row counts:")
print(f"  gold_policy_period_features ........... {df_features.count():,}")
print(f"  gold_policy_period_loss ............... {df_loss.count():,}")
print(f"  gold_expected_loss_scores ............. {df_scores.count():,}")
print(f"  gold_policy_premium_recommendation .... {df_premium.count():,}")

StatementMeta(, 872b224a-3eca-44bf-aa16-e8da0d775338, 3, Finished, Available, Finished, False)

Row counts:
  gold_policy_period_features ........... 82
  gold_policy_period_loss ............... 43
  gold_expected_loss_scores ............. 82
  gold_policy_premium_recommendation .... 82


## Build the joined view via SQL

**Column resolution for duplicates:**

| Column | Appears in | Taken from | Rationale |
|---|---|---|---|
| `coverage_type` | features, premium | features | dimensional attribute lives with features |
| `current_premium` | features, premium | features | original premium lives with policy features |
| `expected_loss_cost` | scores, premium | scores (authoritative), premium kept as `premium_expected_loss_cost` for audit |
| `feature_version` | features, scores, premium | kept per-source with suffix |
| `model_version` | scores, premium | kept per-source with suffix |
| `computed_ts` / `scoring_ts` | all four | kept per-source with suffix |

In [2]:
# -------------------------------------------------------------------
# Cell 4 – Create / replace the materialized view as a Delta table
# -------------------------------------------------------------------

spark.sql("""
CREATE OR REPLACE MATERIALIZED LAKE VIEW mlv_gold_scored_policy_period
AS
SELECT
    -- Unique key  (deterministic hash of the composite natural key)
    -- ═══════════════════════════════════════════════════════════════
    sha2(concat_ws('|', f.policy_number,
                        CAST(f.policy_start_date AS STRING),
                        CAST(f.policy_end_date   AS STRING)), 256)
                                 AS scored_policy_period_key,

    -- ═══════════════════════════════════════════════════════════════
    -- Composite natural key
    -- ═══════════════════════════════════════════════════════════════
    f.policy_number,
    f.policy_start_date,
    f.policy_end_date,

    -- ═══════════════════════════════════════════════════════════════
    -- From gold_policy_period_features  (UBI features & policy dims)
    -- ═══════════════════════════════════════════════════════════════
    f.policyholder_id,
    f.vehicle_vin,
    f.coverage_type,
    f.current_premium,

    f.total_trips,
    f.total_miles,
    f.night_miles_share,
    f.avg_safety_score,
    f.speeding_per_100_miles,
    f.harsh_events_per_100_miles,
    f.high_risk_trip_share,

    f.vehicle_year,
    f.vehicle_make,
    f.vehicle_model,

    --f.feature_version           AS features_feature_version,
    f.computed_ts               AS features_computed_ts,

    -- ═══════════════════════════════════════════════════════════════
    -- From gold_policy_period_loss  (actual loss / claims truth)
    -- ═══════════════════════════════════════════════════════════════
    l.claims_count,
    l.total_claim_amount,
    l.total_payout_amount,
    l.accidents_count,
    l.high_severity_accidents,

    l.label_version,
    l.computed_ts               AS loss_computed_ts,

    -- ═══════════════════════════════════════════════════════════════
    -- From gold_expected_loss_scores  (ML model output)
    -- ═══════════════════════════════════════════════════════════════
    s.risk_score,
    s.expected_loss_cost,
    s.model_name,

    --s.model_version             AS scores_model_version,
    --s.feature_version           AS scores_feature_version,
    s.scoring_ts                AS scores_scoring_ts,

    -- ═══════════════════════════════════════════════════════════════
    -- From gold_policy_premium_recommendation  (pricing output)
    -- ═══════════════════════════════════════════════════════════════
    r.target_loss_ratio,
    r.expense_load,
    r.profit_load,

    r.recommended_technical_premium,
    r.recommended_premium,
    r.premium_change_pct,

    r.actual_loss_ratio,
    r.underpriced_flag,

    r.cap_applied_flag,
    r.smoothing_applied_flag,

    r.reason_codes,
    r.risk_band,

    --r.model_version             AS premium_model_version,
    --r.feature_version           AS premium_feature_version,
    r.scoring_ts                AS premium_scoring_ts

FROM gold_policy_period_features        f
LEFT JOIN gold_policy_period_loss       l
       ON l.policy_number    = f.policy_number
      AND l.policy_start_date = f.policy_start_date
      AND l.policy_end_date   = f.policy_end_date
LEFT JOIN gold_expected_loss_scores     s
       ON s.policy_number    = f.policy_number
      AND s.policy_start_date = f.policy_start_date
      AND s.policy_end_date   = f.policy_end_date
LEFT JOIN gold_policy_premium_recommendation r
       ON r.policy_number    = f.policy_number
      AND r.policy_start_date = f.policy_start_date
      AND r.policy_end_date   = f.policy_end_date
""")

print("✅ gold_scored_policy_period created successfully.")

StatementMeta(, 872b224a-3eca-44bf-aa16-e8da0d775338, 4, Finished, Available, Finished, False)

✅ gold_scored_policy_period created successfully.


In [3]:
# -------------------------------------------------------------------
# Cell 5 – Quick validation
# -------------------------------------------------------------------

df_scored = spark.table("mlv_gold_scored_policy_period")

print(f"Total rows : {df_scored.count():,}")
print(f"Total cols : {len(df_scored.columns)}")
print(f"\nColumn list:")
for i, c in enumerate(df_scored.columns, 1):
    print(f"  {i:>2}. {c}")

StatementMeta(, 872b224a-3eca-44bf-aa16-e8da0d775338, 5, Finished, Available, Finished, False)

Total rows : 82
Total cols : 43

Column list:
   1. scored_policy_period_key
   2. policy_number
   3. policy_start_date
   4. policy_end_date
   5. policyholder_id
   6. vehicle_vin
   7. coverage_type
   8. current_premium
   9. total_trips
  10. total_miles
  11. night_miles_share
  12. avg_safety_score
  13. speeding_per_100_miles
  14. harsh_events_per_100_miles
  15. high_risk_trip_share
  16. vehicle_year
  17. vehicle_make
  18. vehicle_model
  19. features_computed_ts
  20. claims_count
  21. total_claim_amount
  22. total_payout_amount
  23. accidents_count
  24. high_severity_accidents
  25. label_version
  26. loss_computed_ts
  27. risk_score
  28. expected_loss_cost
  29. model_name
  30. scores_scoring_ts
  31. target_loss_ratio
  32. expense_load
  33. profit_load
  34. recommended_technical_premium
  35. recommended_premium
  36. premium_change_pct
  37. actual_loss_ratio
  38. underpriced_flag
  39. cap_applied_flag
  40. smoothing_applied_flag
  41. reason_codes
  42

In [4]:
# -------------------------------------------------------------------
# Cell 6 – Preview sample rows
# -------------------------------------------------------------------

display(df_scored)

StatementMeta(, 872b224a-3eca-44bf-aa16-e8da0d775338, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f56413e-a872-432b-be1d-2bace1567595)

In [5]:
# -------------------------------------------------------------------
# Cell 7 – Null coverage check (detect unmatched joins)
# -------------------------------------------------------------------

from pyspark.sql import functions as F

null_audit = df_scored.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("claims_count").isNull(), 1).otherwise(0)).alias("missing_loss"),
    F.sum(F.when(F.col("risk_score").isNull(), 1).otherwise(0)).alias("missing_scores"),
    F.sum(F.when(F.col("recommended_premium").isNull(), 1).otherwise(0)).alias("missing_premium"),
)

null_audit.show(truncate=False)

StatementMeta(, 872b224a-3eca-44bf-aa16-e8da0d775338, 7, Finished, Available, Finished, False)

+----------+------------+--------------+---------------+
|total_rows|missing_loss|missing_scores|missing_premium|
+----------+------------+--------------+---------------+
|82        |39          |0             |0              |
+----------+------------+--------------+---------------+

